In [ ]:
import sys
from pathlib import Path

import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import chi2_contingency

data_path = "../data/raw/mental-heath-in-tech-2016_20161114.csv"
FIGURES_PATH = "../reports/figures/data_preparation"

sns.set_theme(style="whitegrid", context="notebook")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

warnings.filterwarnings("default", category=FutureWarning)

In [ ]:
name_mapping = {
    "Are you self-employed?": "self_employed",
    "How many employees does your company or organization have?": "company_size",
    "Is your employer primarily a tech company/organization?": "tech_company",
    "Is your primary role within your company related to tech/IT?": "tech_role",
    "Does your employer provide mental health benefits as part of healthcare coverage?": "mh_benefits",
    "Do you know the options for mental health care available under your employer-provided coverage?": "mh_options_awareness",
    "Has your employer ever formally discussed mental health (for example, as part of a wellness campaign or other official communication)?": "mh_formal_discussion",
    "Does your employer offer resources to learn more about mental health concerns and options for seeking help?": "mh_resources_offered",
    "Is your anonymity protected if you choose to take advantage of mental health or substance abuse treatment resources provided by your employer?": "mh_anonymity_protected",
    "If a mental health issue prompted you to request a medical leave from work, asking for that leave would be:": "mh_medical_leave_ease",
    "Do you think that discussing a mental health disorder with your employer would have negative consequences?": "mh_discussion_consequences",
    "Do you think that discussing a physical health issue with your employer would have negative consequences?": "ph_discussion_consequences",
    "Would you feel comfortable discussing a mental health disorder with your coworkers?": "mh_comfort_coworkers",
    "Would you feel comfortable discussing a mental health disorder with your direct supervisor(s)?": "mh_comfort_supervisor",
    "Do you feel that your employer takes mental health as seriously as physical health?": "mh_vs_ph_seriousness",
    "Have you heard of or observed negative consequences for co-workers who have been open about mental health issues in your workplace?": "observed_mh_consequences",
    "Do you have medical coverage (private insurance or state-provided) which includes treatment of \xa0mental health issues?": "medical_coverage_mh",
    "Do you know local or online resources to seek help for a mental health disorder?": "know_external_resources",
    "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to clients or business contacts?": "reveal_mh_clients",
    "If you have revealed a mental health issue to a client or business contact, do you believe this has impacted you negatively?": "client_reveal_neg_impact",
    "If you have been diagnosed or treated for a mental health disorder, do you ever reveal this to coworkers or employees?": "reveal_mh_coworkers",
    "If you have revealed a mental health issue to a coworker or employee, do you believe this has impacted you negatively?": "coworker_reveal_neg_impact",
    "Do you believe your productivity is ever affected by a mental health issue?": "mh_productivity_affected",
    "If yes, what percentage of your work time (time performing primary or secondary job functions) is affected by a mental health issue?": "mh_productivity_pct",
    "Do you have previous employers?": "has_previous_employers",
    "Have your previous employers provided mental health benefits?": "prev_mh_benefits",
    "Were you aware of the options for mental health care provided by your previous employers?": "prev_mh_awareness",
    "Did your previous employers ever formally discuss mental health (as part of a wellness campaign or other official communication)?": "prev_mh_discussion",
    "Did your previous employers provide resources to learn more about mental health issues and how to seek help?": "prev_mh_resources",
    "Was your anonymity protected if you chose to take advantage of mental health or substance abuse treatment resources with previous employers?": "prev_mh_anonymity",
    "Do you think that discussing a mental health disorder with previous employers would have negative consequences?": "prev_mh_consequences",
    "Do you think that discussing a physical health issue with previous employers would have negative consequences?": "prev_ph_consequences",
    "Would you have been willing to discuss a mental health issue with your previous co-workers?": "prev_willingness_coworkers",
    "Would you have been willing to discuss a mental health issue with your direct supervisor(s)?": "prev_willingness_supervisor",
    "Did you feel that your previous employers took mental health as seriously as physical health?": "prev_mh_vs_ph_seriousness",
    "Did you hear of or observe negative consequences for co-workers with mental health issues in your previous workplaces?": "prev_observed_mh_consequences",
    "Would you be willing to bring up a physical health issue with a potential employer in an interview?": "interview_ph_reveal",
    "Why or why not?": "interview_ph_reveal_why",
    "Would you bring up a mental health issue with a potential employer in an interview?": "interview_mh_reveal",
    "Why or why not?.1": "interview_mh_reveal_why",
    "Do you feel that being identified as a person with a mental health issue would hurt your career?": "mh_hurt_career",
    "Do you think that team members/co-workers would view you more negatively if they knew you suffered from a mental health issue?": "coworker_negative_view",
    "How willing would you be to share with friends and family that you have a mental illness?": "share_with_friends_family",
    "Have you observed or experienced an unsupportive or badly handled response to a mental health issue in your current or previous workplace?": "observed_bad_mh_handling",
    "Have your observations of how another individual who discussed a mental health disorder made you less likely to reveal a mental health issue yourself in your current workplace?": "observed_bad_handling_impact",
    "Do you have a family history of mental illness?": "family_history_mh",
    "Have you had a mental health disorder in the past?": "past_mh_disorder",
    "Do you currently have a mental health disorder?": "current_mh_disorder",
    "If yes, what condition(s) have you been diagnosed with?": "current_diagnosed_conditions",
    "If maybe, what condition(s) do you believe you have?": "suspected_conditions",
    "Have you been diagnosed with a mental health condition by a medical professional?": "diagnosed_by_pro",
    "If so, what condition(s) were you diagnosed with?": "pro_diagnosed_conditions",
    "Have you ever sought treatment for a mental health issue from a mental health professional?": "sought_mh_treatment",
    "If you have a mental health issue, do you feel that it interferes with your work when being treated effectively?": "mh_interference_treated",
    "If you have a mental health issue, do you feel that it interferes with your work when NOT being treated effectively?": "mh_interference_untreated",
    "What is your age?": "age",
    "What is your gender?": "gender",
    "What country do you live in?": "country_live",
    "What US state or territory do you live in?": "us_state_live",
    "What country do you work in?": "country_work",
    "What US state or territory do you work in?": "us_state_work",
    "Which of the following best describes your work position?": "work_position",
    "Do you work remotely?": "work_remote"
}


In [ ]:
environment = pd.DataFrame({
    "component": [
        "Python",
        "pandas",
        "NumPy",
        "Matplotlib",
        "Seaborn",
    ],
    "version": [
        platform.python_version(),
        pd.__version__,
        np.__version__,
        __import__("matplotlib").__version__,
        sns.__version__,
    ],
})

display(environment)

print("Project root:", PROJECT_ROOT)
print("Raw directiory:", RAW_DIR)
print("Processed directory:", PROCESSED_DIR)
print("Preparation directory:", PREPARATION_DIR)
print("Figures directory:", FIGURES_DIR)
print("Random state:", RANDOM_STATE)

## 3. Source Data Integrity and Provenance

The original CSV is the immutable source of record. Its filename, dimensions, file size, and SHA-256 checksum establish the identity of the data used in this phase.

The source is loaded independently of the Data Understand notebook. No variables or intermediate DataFrames are inherited from the previous notebook's kernel.

A checksum is calculated over the file's bytes:

$$
H = \operatorname{SHA256}(\text{source file})
$$

The checksum provides a reproducible content fingerprint. A change in the source file, even if its filename remains unchanged, produces a different checksum.

In [ ]:
assert SOURCE_PATH.exists(), (
    f"Source file not found: {SOURCE_PATH}"
)

df_raw, source_metadata = load_source(
    SOURCE_PATH,
    expected_shape=EXPECTED_RAW_SHAPE,
)

source_metadata["sha256"] = sha256_file(SOURCE_PATH)
source_metadata["python_version"] = platform.python_version()
source_metadata["pandas_version"] = pd.__version__
source_metadata["numpy_version"] = np.__version__

source_metadata_table = pd.DataFrame(
    [
        {"property": key, "value": str(value)}
        for key, value in source_metadata.items()
    ]
)

display(source_metadata_table)

assert df_raw.shape == EXPECTED_RAW_SHAPE
assert df_raw.shape == (1433, 63)

### 3.1 Source Integrity Findings

The supplied source has 1,433 rows and 63 columns, consistent with the dimensions established during Data Understanding.

The source is retained as `df_raw`. All subsequent operations produce derived objects. This separation allows the cleaned data to be traced back to the original survey responses and prevents transformations from silently changing the source.

## 4. Raw Data Structure and Respondent Lineage

In [ ]:
raw_structure = pd.DataFrame({
    "column_position": np.arange(1, len(df_raw.columns) + 1),
    "source_column": df_raw.columns,
    "stored_dtype": df_raw.dtypes.astype(str).values,
    "observed": df_raw.notna().sum().values,
    "missing": df_raw.isna().sum().values,
    "missing_percentage": (
        df_raw.isna().mean().values * 100
    ).round(2),
    "distinct_non_missing": (
        df_raw.nunique(dropna=True).values
    ),
})

display(raw_structure)

In [ ]:
raw_integrity = pd.DataFrame({
    "measure": [
        "Rows",
        "Columns",
        "Exact duplicate rows",
        "Completely empty rows",
        "Completely empty columns",
        "Total missing cells",
        "Total observed cells",
    ],
    "value": [
        len(df_raw),
        df_raw.shape[1],
        int(df_raw.duplicated().sum()),
        int(df_raw.isna().all(axis=1).sum()),
        int(df_raw.isna().all(axis=0).sum()),
        int(df_raw.isna().sum().sum()),
        int(df_raw.notna().sum().sum()),
    ],
})

display(raw_integrity)

### 4.1 Row-Integrity Decision

The source does not contain a respondent identifier suitable for linking an individual to an external identity. A preparation-stage row key is therefore generated from the original row order.

The key is a technical lineage field, not a respondent attribute. It is excluded from the analytical feature matrix and is retained only in internal row-order and audit artefacts.

Exact duplicate rows are assessed across all source columns. Similar response patterns are not treated as duplicates because the survey does not provide evidence that they represent the same individual.

In [ ]:
df_work = df_raw.copy()

df_work.insert(
    0,
    "source_row_id",
    np.arange(len(df_work), dtype=np.int64),
)

assert df_work["source_row_id"].is_unique
assert df_work["source_row_id"].dtype == np.dtype("int64")

print("Source rows with unique lineage identifiers:", len(df_work))

## 5. Canonical Schema and Semantic Column Mapping

The source uses complete survey questions as column names. These names are preserved in the source dataset but are not convenient for reusable analytical code.

The repository's schema module maps the original question wording to stable canonical names. This mapping improves readability while retaining a complete record of the original-to-canonical relationship.

The mapping is not a feature-selection operation. It changes column labels, not the underlying response values.

In [ ]:
df_canonical, column_mapping_audit = rename_columns(df_work)

display(column_mapping_audit)

assert df_canonical.columns.is_unique
assert "source_row_id" in df_canonical.columns

validate_model_schema(df_canonical)

print("Canonical dataset shape:", df_canonical.shape)
print("Canonical columns:", df_canonical.shape[1])

In [ ]:
canonical_schema = pd.DataFrame({
    "canonical_column": df_canonical.columns,
    "dtype": df_canonical.dtypes.astype(str).values,
    "observed": df_canonical.notna().sum().values,
    "missing": df_canonical.isna().sum().values,
    "distinct": df_canonical.nunique(dropna=True).values,
})

display(canonical_schema)

### 5.1 Schema Validation

The canonical schema is validated against the 15-feature specification in the repository configuration. This ensures that every approved primary feature has a corresponding source variable.

The mapping audit remains available as a traceability artefact. It allows a reviewer to move from a short feature name back to the exact survey question without relying on column position or memory.

## 6. Reconstructing and Validating the Eligible Cohort

The cohort definition established in the Data Understanding is applied exactly:

$$
E_i = 
(S_i = 0)
\land
(T_i = 1 \lor R_i = 1)
$$

where:

- $(S_i)$ is the self-employment response;
- $(T_i)$ indicates whether the employer is primarily a technology organisation;
- $(R_i)$ indicates whether the respondent's primary role is related to technology/IT;
- $(E_I)$ indicates eligibility for the primary cohort.

The technology-role field is conditional. Its missing values are not interpreted as negative responses. The logical OR in the approved cohort rule is applied to the observed source coding without imputing the conditional field.

This cohort rule retains non-self-employed respondents who satisfy either technology-related criterion.

In [ ]:
self_employed = df_canonical["self_employed"]
tech_employer = df_canonical["tech_employer"]
tech_role = df_canonical["tech_role"]

cohort_mask = (
    self_employed.eq(0)
    & (
        tech_employer.eq(1)
        | tech_role.eq(1)
    )
)

cohort = df_canonical.loc[cohort_mask].copy()

cohort_flow = pd.DataFrame({
    "stage": [
        "All survey responses",
        "Non-self-employed respondents",
        "Eligible technology-related respondents",
        "Excluded from primary cohort",
    ],
    "respondents": [
        len(df_canonical),
        int(self_employed.eq(0).sum()),
        int(cohort_mask.sum()),
        int((~cohort_mask).sum()),
    ],
})

cohort_flow["percentage_of_source"] = (
    cohort_flow["respondents"] / len(df_canonical) * 100
).round(2)

display(cohort_flow)

assert len(cohort) == EXPECTED_COHORT_SIZE
assert len(cohort) == 1131
assert cohort["source_row_id"].is_unique

### 6.1 Cohort Interpretation

The supplied data contain 1,433 responses. Of these, 1,146 respondents are classified as non-self-employed and 287 as self-employed.

Applying the approved technology-related eligibility rule produces the expected 1,131-person cohort.

The difference between the 1,146 non-self-employed respondents and the 1,131-person final cohort is determined by the technology-related condition, not by deleting respondents with missing decriptive variables.

The cohort is therefore established before the preparation of the primary feature matrix. Subsequent cleaning and encoding do not alter cohort membership.

In [ ]:
validate_cohort(cohort)

assert cohort["source_row_id"].isin(
    df_canonical["source_row_id"]
).all()

print("Cohort validation passed.")
print("Eligible repondents:", len(cohort))
print(
    "Unique source row identifiers:",
    cohort["source_row_id"].nunique()
)

## 7. Text Normalisation and Response Harmonisation

Survey responses contain formatting differences caused by whitespace, Unicode representation, and inconsistent text entry. These differences can create artificial categories even when the underlying response is identical.

The cleaning process applies conservative normalisation:

- Unicode compatibility normalisation;
- replacement of non-breaking spaces;
- removal of zero-width and non-printing characters where applicable;
- standardisation of repeated whitespace;
- removal of leading and trailing whitespace;
- conversion of empty strings to missing values.

The process does not collapse semantically distinct responses. In particular:

- `Maybe` remains distinct from `Yes` and `No`;
- `I don't know` remains distinct from `No`;
- `Not eligible for coverage / N/A` remains distinct from an actual negative response;
- missing values remian missing.

This distinction is essential because uncertainty, non-applicability, and nonresponse describe different states.

In [ ]:
df_clean = normalize_string_columns(df_canonical)

categorical_columns = (
    df_clean
    .select_dtypes(include=["object", "string"])
    .columns
    .tolist()
)

df_clean =  harmonise_categorical_columns(
    df_clean,
    categorical_columns,
)

print("Text columns harmonised:", len(categorical_columns))

In [ ]:
response_columns = [
    feature
    for feature in MODEL_FEATURES
]

response_category_audit = []

for feature in response_columns:
    counts = (
        df_clean.loc[cohort.index, feature]
        .value_counts(dropna=False)
    )

    for response, count in counts.items():
        response_category_audit.append({
            "feature": feature,
            "response": (
                "<MISSING>" if pd.isna(response) else str(response)
            ),
            "count": int(count),
            "percentage": round(100 * count / len(cohort), 2),
        })

response_category_audit = pd.DataFrame(
    response_category_audit
)

display(response_category_audit)

### 7.1 Response Harmonisation Interpretation

Response normalisation is deliberately conservative. The cleaned response categories retain the distinctions required by the survey's meaning.

No global binary recoding is applied to the 15 primary features. Several questions contain uncertainty or non-applicability categories, so treating every response as a simple Yes/No variable would destroy information.

The category audit provides a complete record of the response distribution within the eligible cohort.

## 8. Missingness Analysis and Applicability

Missingness is interpreted in the context of the survey's branching structure.

A blank response may arise because:

1. The respondent was not eligible to answer the question.
2. The survey routed the respondent away from a conditional follow-up.
3. The respondent was eligible but did not provide an answer.
4. The reason for the blank cannot be established from the available data.

These conditions must not be treated as equivalent.

The primary cohort is intended to make the 15 selected current-workplace variables applicable. Their completeness is verified directly. The benefit-awareness follow-up and the supplementary unsupportive-response item remain outside the primary feature matrix and retain their own denominators.

In [ ]:
source_missingness = build_missingness_audit(df_clean)
cohort_missingness = build_missingness_audit(cohort)

display(source_missingness.head(30))

In [ ]:
applicability_register = build_applicability_register(cohort)

display(applicability_register)

In [ ]:
primary_missingness = (
    cohort[MODEL_FEATURES]
    .isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)

primary_missingness["observed_count"] = (
    len(cohort) - primary_missingness["missing_count"]
)

primary_missingness["missing_percentage"] = (
    primary_missingness["missing_count"] / len(cohort) * 100
)

display(primary_missingness)

assert_primary_features_complete(cohort)

### 8.1 Missingness Decision

The primary 15-feature matrix is complete in the eligible cohort. No imputation is required for these 15 variables.

This is an important distinction: completeness is achieved through the approved population definition and the selection of applicable workplace questions, not through replacing unanswered responses with assumed values.

The supplementary unsupportive-response variable is also kept outside the primary matrix. Its missingness is retained and documented rather than recoded as an absence of an unsupportive experience.

In [ ]:
supplementary_audit = []

for feature in SUPPLEMENTARY_FEATURES:
    if feature not in cohort.columns:
        continue

    series = cohort[feature]

    supplementary_audit.append({
        "feature": feature,
        "eligible_cohort_n": len(cohort),
        "observed_n": int(series.notna().sum()),
        "missing_n": int(series.isna().sum()),
        "observed_percentage": round(100 * series.notna().mean(), 2),
        "missing_percentage": round(100 * series.isna().mean(), 2),
        "distinct_observed": int(series.nunique(dropna=True)),
    })

supplementary_audit = pd.DataFrame(supplementary_audit)

display(supplementary_audit)

## 9. Age Quality Assessment, Cleaning, and Imputation

Age is retained as a descriptive profile variable. It is not part of the primary 15-feature matrix.

The Data Understanding phase identified five implausible reported values:

| Reported age | Quality assessment |
|---:|---|
| 3 | Implausible for the survey's adult respondent population |
| 15 | Outside the intended adult respondent population |
| 17 | Outside the intended adult respondent population |
| 99 | Outside the approved plausible-age range |
| 323 | Impossible age; likely data-entry error |

The approved plausible range is 18-75 years. Values outside this interval are invalidated in the derived cleaned-age field. The original reported values remain preserved in the raw source.

Because age is retained for descriptive profiling, the cleaned age descriptor is completed using the median age of the eligible cohort. The median is selected because it is resistant to extreme observations and does not require a normal-distribution assumption.

For valid cleaned ages $(a_1, \ldots, a_n)$, the imputed value is:

$$
\tilde{a} = \operatorname{median}(a_1, \ldots, a_n)
$$

The imputation is performed only after invalid ages have been converted to missing values. A separate indicator distinguishes observed valid ages from imputed values.

This operation does not change cohort membership and does not introduce age into the primary feature matrix.

In [ ]:
cohort_age_cleaned, age_cleaning_audit = clean_age(
    cohort,
    source_column="age",
    cleaned_column="age_clean",
)

age_cleaning_audit = age_cleaning_audit.merge(
    cohort[["source_row_id"]],
    left_on="source_row_id",
    right_on="source_row_id",
    how="left",
)

display(age_cleaning_audit)

In [ ]:
age_before_imputation = cohort_age_cleaned["age_clean"].copy()

age_quality_summary = pd.DataFrame({
    "measure": [
        "Eligible respondents",
        "Valid observed ages",
        "Missing ages after plausibility screening",
        "Median valid age",
        "Mean valid age",
        "First quartile",
        "Third quartile",
        "Minimum valid age",
        "Maximum valid age",
    ],
    "value": [
        len(cohort_age_cleaned),
        int(age_before_imputation.notna().sum()),
        int(age_before_imputation.isna().sum()),
        age_before_imputation.median(),
        round(age_before_imputation.mean(), 2),
        age_before_imputation.quantile(0.25),
        age_before_imputation.quantile(0.75),
        age_before_imputation.min(),
        age_before_imputation.max(),
    ],
})

display(age_quality_summary)

In [ ]:
age_imputation_value = int(
    age_before_imputation.median()
)

cohort_age_cleaned["age_was_imputed"] = (
    age_before_imputation.isna().astype("Int8")
)

cohort_age_cleaned["age_imputed"] = (
    age_before_imputation
    .fillna(age_imputation_value)
    .astype("Int64")
)

cohort_age_cleaned["age_implausible_flag"] = (
    cohort_age_cleaned["age_implausible_flag"]
    .astype("Int8")
)

assert cohort_age_cleaned["age_imputed"].notna().all()
assert pd.api.types.is_integer_dtype(
    cohort_age_cleaned["age_imputed"].dtype
)
assert pd.api.types.is_integer_dtype(
    cohort_age_cleaned["age_was_imputed"].dtype
)

age_imputation_summary = pd.DataFrame({
    "measure": [
        "Eligible cohort size",
        "Valid observed ages before imputation",
        "Values imputed",
        "Median used for imputation",
        "Missing ages after imputation",
    ],
    "value": [
        len(cohort_age_cleaned),
        int(age_before_imputation.notna().sum()),
        int(cohort_age_cleaned["age_was_imputed"].sum()),
        age_imputation_value,
        int(cohort_age_cleaned["age_imputed"].isna().sum()),
    ],
})

display(age_imputation_summary)

### 9.1 Age Imputation Interpretation

The cleaned age distribution is based on the approved 18-75 range. The invalid observations are represented as missing in `age_clean`, while the source values remain unchanged in the original dataset.

Within the eligible cohort, four invalid age observations require imputation. The observed valid-age median is 32 years. The derived `age_imputed` field therefore uses 32 for those four records.

The imputation indicator, `age_was_imputed`, identifies these records explicitly. The cleaned non-imputed field, `age_clean`, remains available for analyses that require only observed valid ages.

The descriptive age field is now complete without deleting respondents or changing the primary feature representation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

age_before_imputation.dropna().plot(
    kind="hist",
    bins=20,
    ax=axes[0],
    edgecolor="black",
)

axes[0].set_title("Observed Valid Age")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Respondents")

cohort_age_cleaned["age_imputed"].plot(
    kind="hist",
    bins=20,
    ax=axes[1],
    edgecolor="black",
)

axes[1].set_title("Age Descriptor after Median Imputation")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Respondents")

plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "data_preparation_age_imputation.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 10. Gender and Contextual Descriptor Cleaning

Gender was collected as free text. The source contains inconsistent capitalisation, abbreviations, spelling variations, and self-described identities.

The repository's cleaning rules standardise clearly equivalent responses into the agreed descriptive groups while retaining the original response in `gender_original`. Gender is not a primary feature.

Country and state descriptors receive conservative whitespace and Unicode normalisation. The residence and workplace geography fields remain distinct because residence location and work location are not interchangeable.

Employer size, technology context, remote-work arrangement, and job position remain descriptive context. They are not added to the primary 15-feature matrix.

In [ ]:
cohort_context = cohort_age_cleaned.copy()

# Retain the original gender response before applying the approved mapping.
cohort_context["gender_original"] = cohort_context["gender"]

cohort_context = clean_gender(cohort_context)
cohort_context = clean_countries(cohort_context)

gender_audit = (
    cohort_context["gender"]
    .value_counts(dropna=False)
    .rename_axis("clean_gender")
    .reset_index(name="respondents")
)

gender_audit["percentage"] = (
    round(
        gender_audit["respondents"]
        / len(cohort_context) * 100, 2
    )
)

display(gender_audit)

In [ ]:
context_columns = [
    column
    for column in PROFILE_DESCRIPTORS
    if column in cohort_context.columns
]

profile_descriptors = build_profile_descriptors(cohort_context)

for column in [
    "age_imputed",
    "age_was_imputed",
    "gender_original",
    "age_implausible_flag",
]:
    if column in cohort_context.columns:
        profile_descriptors[column] = cohort_context[column]

profile_descriptors.insert(
    0,
    "source_row_id",
    cohort_context["source_row_id"].to_numpy(dtype=np.int64),
)

display(profile_descriptors.head())

In [ ]:
descriptor_quality = pd.DataFrame({
    "descriptor": profile_descriptors.columns,
    "dtype": profile_descriptors.dtypes.astype(str).values,
    "observed": profile_descriptors.notna().sum().values,
    "missing": profile_descriptors.isna().sum().values,
    "distinct": profile_descriptors.nunique(dropna=True).values,
})

display(descriptor_quality)

## 11. Primary Feature Selection and Exclusion Governance

The primary feature matrix is restricted to the 15 varaibles specified in the approved study design. The three conceptual domains are preserved exactly as defined in the repository configuration.

The following variable groups remain outside the primary matrix:

- Descriptive demographic and employment-context varaibles.
- Supplementary workplace questions with distinct applicability or missingness.
- Personal mental-health history, diagnosis, treatment, and condition-description variables.
- Free-text survey responses.
- Technical identifiers and source-row lineage fields.

The exclusion of personal-health information is a substantive governance decision. It prevents the workplace-profile representation from becoming a proxy for an individual's disclosed health history.

The feature decision register documents the role assigned to every canonical column.

In [ ]:
feature_decision_register = build_feature_decision_register(
    cohort_context
)

display(feature_decision_register[
    feature_decision_register["decision"]
    .ne("excluded_or_unassigned")
])

In [ ]:
feature_matrix = select_model_features(cohort_context)

validate_feature_exclusions(feature_matrix)

assert feature_matrix.shape == (1131, 15)
assert list(feature_matrix.columns) == MODEL_FEATURES

display(feature_matrix.head)

In [ ]:
domain_membership = build_domain_metadata()

display(domain_membership)

domain_summary = (
    domain_membership
    .groupby("domain", as_index=False)
    .agg(
        feature_count=("feature", "count"),
        domain_weights=("domain_weight", "first"),
        within_domain_weight=("within_domain_feature_weight", "first"),
    )
)

display(domain_summary)

assert domain_membership["feature"].nunique() == 15
assert set(domain_membership["feature"]) == set(MODEL_FEATURES)

### 11.1 Feature-Selection Interpretation

The primary matrix contains exactly 15 variables. The feature membership is determined by the approved domain specification rather than by automatic selection based on correlation or response frequency.

The domains contain four, seven, and four features respectively. Each domain receives one-third of the total representation weight, with equal contribution from its constituent features within that domain.

The profile-descriptor table is stored separately. This separation ensures that demographic, contextual, supplementary, and sensitive variables cannot enter the primary matrix through an accidental concatenation.

## 12. Category Quality, Response Semantics, and Encoding

The feature matrix contains nominal and ordinal categorical responses. Their representation must reflect their measurement meaning.

Nominal responses have no defensible ranking. Their original labels are retained, and they are not converted into arbitrary integer codes.

Ordinal responses have a meaningful order. Their ranks are stored in nullable integer columns. The nullable integer type preserves missingness without converting integer codes into floating-point values.

For the seven ordered items specified by the approved response design, the encoding register records the category-to-integer mapping. The medical-leave item has five substantive ordered responses. Its explicit `I don't know` category is not assigned an artificial rank; it is retained separately.

The actual cohort also contains `I don't know` for `mh_as_serious_as_physical`. This response is not an ordered point between `No` and `Yes`. That item is therefore treated as nominal, preserving all three responses as distinct categories.

In [ ]:
encoded_data, ordinal_encoding_audit = encode_ordinal_features(cohort_context)
encoded_data, medical_leave_encoding_audit = encode_medical_leave(encoded_data)

# Store the unknown-response indicator as an integer code, not float or bool.
encoded_data["medical_leave_ease_unknown"] = (
    encoded_data["medical_leave_ease_unknown"]
    .astype("Int8")
)

# Retain the age-imputation flags as integer indicators.
encoded_data["age_was_imputed"] = (
    encoded_data["age_was_imputed"]
    .astype("Int8")
)

encoded_data["age_implausible_flag"] = (
    encoded_data["age_implausible_flag"]
    .astype("Int8")
)

validate_encoded_integer_types(encoded_data)

display(ordinal_encoding_audit)
display(medical_leave_encoding_audit)

In [ ]:
encoded_columns = [
    column
    for column in encoded_data.columns
    if column.endswith("_ordinal")
    or column in {
        "medical_leave_ease_unknown",
        "age_was_imputed",
        "age_implausible_flag",
    }
]

encoding_dtype_audit = pd.DataFrame({
    "encoded_column": encoded_columns,
    "dtype": [
        str(encoded_data[column].dtype)
        for column in encoded_columns
    ],
    "is_integer_dtype": [
        pd.api.types.is_integer_dtype(encoded_data[column].dtype)
        for column in encoded_columns
    ],
    "missing": [
        int(encoded_data[column].isna().sum())
        for column in encoded_columns
    ],
    "distinct": [
        int(encoded_data[column].nunique(dropna=True))
        for column in encoded_columns
    ],
})

display(encoding_dtype_audit)

assert encoding_dtype_audit["is_integer_dtype"].all()
assert not any(
    "float" in dtype
    for dtype in encoding_dtype_audit["dtype"]
)

### 12.1 Encoding Interpretation

Ordinal codes are stored using nullable integer dtypes (`Int8`). No ordinal code is stored as a floating-point value.

The integer values represent category order, not measured distances in the original survey. The distance calculation uses the explicitly documented ordinal treatment rather than assuming that arbitrary numeric labels are meaningful.

Nominal responses remain categorical strings. This preserves the distinction between categories such as `No`, `Yes`, `I don't know`, and `Not eligible for coverage / N/A`.

The medical-leave uncertainty indicator is retained as a separate integer field for auditability. It is not added as a sixteenth primary feature; the original medical-leave response remains the single primary feature.

## 13. Rare-Category and Semantic Redundancy Audit

Category frequencies are examined to identify rare responses and potential data-quality problems. Rare categories are not automatically merged. A low-frequency answer may represent a genuine minority experience, and collapsing it without evidence could remove meaning information.

Association between categorical variables is also examined as a redundancy diagnostic. Cramér's V is used as a descriptive effect-size measure:

$$
V = \sqrt{\frac{\chi^2}{n \times \min(r - 1, c - 1)}}
$$

where:

- **$\chi^2$** is the Pearson chi-square test statistic;
- **$n$** is the number of observations in the contingency table;
- **$r$** is the number of rows in the contingency table;
- **$c$** is the number of columns in the contingency table;
- **$\min(r - 1, c - 1)$** is the smaller value between $(r - 1)$ and $(c - 1)$.

Cramér's V ranges from 0 to 1. Larger values indicate stronger association. Association does not establish that two questions measure the same construct, so the results are interpreted alongside the wording and intended meaning of each survey item.

The audit is decriptive. It does not automatically remove features or alter the predeclared feature domains.

In [ ]:
rare_category_records = []

for feature in MODEL_FEATURES:
    counts = feature_matrix[feature].value_counts(dropna=False)

    for category, count in counts.items():
        rare_category_records.append({
            "feature": feature,
            "category": str(category),
            "count": int(count),
            "percentage": round(100 * count / len(feature_matrix), 4),
            "below_1_percent": (
                round(100 * count / len(feature_matrix) < 1, 6)
            ),
        })

rare_category_audit = pd.DataFrame(rare_category_records)

display(
    rare_category_audit.sort_values(
        ["below_1_percent", "percentage"],
        ascending=[False, True],
    )
)

In [ ]:
def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)

    if min(table.shape) < 2:
        return np.nan

    chi2 = chi2_contingency(
        table,
        correction=False,
    )[0]

    n= table.to_numpy().sum()
    denominator = n * min(table.shape[0] - 1, table.shape[1] - 1)

    if denominator == 0:
        return np.nan

    return float(np.sqrt(chi2 / denominator))

association_matrix = pd.DataFrame(
    np.eye(len(MODEL_FEATURES)),
    index=MODEL_FEATURES,
    columns=MODEL_FEATURES,
)

for i, feature_a in enumerate(MODEL_FEATURES):
    for feature_b in MODEL_FEATURES[i + 1:]:
        value = cramers_v(
            feature_matrix[feature_a],
            feature_matrix[feature_b],
        )

        association_matrix.loc[feature_a, feature_b] = value
        association_matrix.loc[feature_b, feature_a] = value

display(association_matrix.round(3))

In [ ]:
association_pairs = []

for i, feature_a in enumerate(MODEL_FEATURES):
    for feature_b in MODEL_FEATURES[i + 1:]:
        association_pairs.append({
            "feature_a": feature_a,
            "feature_b": feature_b,
            "cramers_v": round(
                association_matrix.loc[feature_a, feature_b], 4,
            ),
        })

association_pairs = (
    pd.DataFrame(association_pairs)
    .sort_values("cramers_v", ascending=False)
    .reset_index(drop=True)
)

display(association_pairs.head(20))

In [ ]:
plt.figure(figsize=(13, 10))

sns.heatmap(
    association_matrix,
    cmap="viridis",
    vmin=0,
    vmax=1,
    annot=True,
    fmt=".2f",
    square=True,
)

plt.xticks(rotation=45, ha="right")
plt.title("Categorical Association between Primary Features")
plt.tight_layout()
plt.savefig(
    FIGURES_DIR / "primary_feature_cramers_v.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

### 13.1 Redundancy Decision

The association matrix identifies pairs of features with stronger categorical relationships. These results are interpreted as evidence of related response patterns, not proof of duplication.

The original survey questions measure distinct contexts, including employer disclosure, coworker disclosure, supervisor disclosure, physical-health comparison, and interview disclosure. Those distinctions are retained unless the evidence demonstrates that a variable is unusable or genuinely duplicates another item.

No category is merged solely because it is rare, and no feature is removed solely because it is associated with another feature. The approved 15-feature specification remains intact.

## 14. Final Primary Feature Matrix

The primary feature matrix is extracted from the eligible cohort in the exact order specified by the repository configuration.

It contains only the 15 approved workplace-response variables. It excludes:

- source row identifiers;
- age and other demographic descriptors;
- employer and job-context descriptors;
- supplementary questions;
- personal mental-health history;
- diagnosis and treatment fields;
- free-text fields;
- ordinal audit columns and auxiliary indicators.

The final matrix retains the cleaned categorical response labels. The additional integer-coded columns are retained separately for auditability and for the approved ordinal distance calculations.

In [ ]:
X_model = select_model_features(cohort_context)

validate_feature_exclusions(X_model)
validate_feature_matrix(X_model)

assert X_model.shape == (EXPECTED_COHORT_SIZE, 15)
assert list(X_model.columns) == MODEL_FEATURES
assert not X_model.isna().any().any()

display(X_model.head())

In [ ]:
feature_matrix_summary = pd.DataFrame({
    "feature": X_model.columns,
    "domain": [
        next(
            domain
            for domain, features in DOMAIN_FEATURES.items()
            if feature in features
        )
        for feature in X_model.columns
    ],
    "dtype": X_model.dtypes.astype(str).values,
    "observed": X_model.notna().sum().values,
    "missing": X_model.isna().sum().values,
    "distinct_categories": X_model.nunique(dropna=True).values,
})

display(feature_matrix_summary)

## 15. Domain-Balanced Dissimilarity Construction

The primary feature matrix contains categorical responses with nominal and ordinal semnantics. The dissimilarity representation therefore applies different treatments according to each variable's measurement meaning.

For a nominal feature:

$$
\delta_{ijk} =
\begin{cases}
0 & \text{if } x_{ik}=x_{jk}\\
1 & \text{otherwise}
\end{cases}
$$

For an ordinal feature with integer ranks from 0 to $(R_k)$:

$$
\delta_{ijk} =
\frac{|r_{ik}-r_{jk}|}{R_k}
$$

The medical-leave question has five substantive ordered categories. Its explicit `I don't know` response is treated as a separate state:

- `I don't know` compared with `I don't know`: dissimilarity 0.
- `I don't know` compared with any substantive category: dissimilarity 1.
- Two substantive categories: normalised absolute rank difference.

The three domains receive equal total weight. Within each domain, features contribute equally.

$$
d(i, j)=
\frac{1}{3}
\sum_{g=1}^{3}
\left[
    \frac{1}{p_g}
    \sum_{k\in g}\delta_{ijk}
\right]
$$

Here, $(p_g)$ is the number of features in domain $(g)$. The domain sizes are 4, 7, and 4.

The final dissimilarity lies in the interval $([0,1])$. A value of 0 indicates identical responses across all 15 features; a value of 1 indicates maximum dissimilarity under the specified feature-level rules.

In [ ]:
validate_domain_weights()

feature_matrices = feature_distance_matrix(X_model)
domain_matrices = domain_distance_matrices(X_model)

D = gower_distance_matrix(X_model)

validate_distance_matrix(D)

assert D.shape == (len(X_model), len(X_model))
assert D.dtype == np.float64
assert np.isfinite(D).all()
assert np.allclose(D, D.T, atol=1e-10)
assert np.allclose(np.diag(D), 0.0, atol=1e-10)
assert D.min() >= -1e-10
assert D.max() <= 1.0 + 1e-10

print("Dissimilarity matrix shape:", D.shape)
print("Dissimilarity matrix dtype:", D.dtype)
print("Minimum dissimilarity:", float(D.min()))
print("Maximum dissimilarity:", float(D.max()))

In [ ]:
domain_distance_summary = []

for domain, matrix in domain_matrices.items():
    upper = matrix[np.triu_indices_from(matrix, k=1)]

    domain_distance_summary.append({
        "domain": domain,
        "feature_count": len(DOMAIN_FEATURES[domain]),
        "domain_weight": DOMAIN_WEIGHTS[domain],
        "mean_pairwise_dissimilarity": float(upper.mean()),
        "median_pairwise_dissimilarity": float(np.median(upper)),
        "minimum_pairwise_dissimilarity": float(upper.min()),
        "maximum_pairwise_dissimilarity": float(upper.max()),
    })

domain_distance_summary = pd.DataFrame(domain_distance_summary)

display(domain_distance_summary)

### 15.1 Dissimilarity Validation

The matrix is assessed against its mathematical requirements:

- It is square and aligned with the 1,131 eligible respondents.
- Every entry is finite.
- It is symmetric.
- It's diagonal is zero.
- No value is negative.
- No value exceeds 1.

The matrix is stored as `float64` because it contains normalised dissimilarities. This is distinct from category encoding: the ordinal response codes remain integer typed, while the calculated pairwise dissimilarities are continuous numerical quantities.

In [ ]:
distance_validation = {
    "shape": D.shape,
    "dtype": str(D.dtype),
    "finite": bool(np.isfinite(D).all()),
    "symmetric": bool(np.allclose(D, D.T, atol=1e-10)),
    "zero_diagonal": bool(np.allclose(np.diag(D), 0.0, atol=1e-10)),
    "non_negative": bool((D >= -1e-10).all()),
    "bounded_above_one": bool((D <= 1.0 + 1e-10).all()),
    "minimum": float(D.min()),
    "maximum": float(D.max()),
    "mean_including_diagonal": float(D.mean()),
    "mean_off_diagnonal": float(D[np.triu_indices_from(D, k=1)].mean()),
}

display(
    pd.DataFrame(
        distance_validation.items(),
        columns=["validation", "result"],
    )
)

assert all([
    distance_validation["finite"],
    distance_validation["symmetric"],
    distance_validation["zero_diagonal"],
    distance_validation["non_negative"],
    distance_validation["bounded_above_one"]
])

## 16. Final Preparation Validation

The final validation consolidates the cohort, feature, encoding, lineage, and dissimilarity checks.

The validation is intended to detect structural preparation errors before the prepared artefacts are consumed by the next analytical phase. It does not evaluate or interpret any discovered respondent groups.

The principal conditions are:

1. The cohort contains exactly 1,131 unique source rows.
2. The primary feature matrix contains exactly 15 columns in the approved order.
3. Every primary feature is complete.
4. The domain specification contains exactly three domains.
5. The domain weights sum to one and are equal.
6. All ordinal encodings use integer dtypes.
7. No descriptive, supplementary, sensitive, or identifier field enters the feature matrix.
8. The dissimilarity matrix is finite, symmetric, zero-diagonal, and bounded.
9. Every cohort row can be traced to the original source.

In [ ]:
validate_cohort(cohort_context)
validate_feature_matrix(X_model)
validate_domain_weights()
validate_integer_codes(encoded_data)
validate_distance(D)
validate_row_lineage(df_canonical, cohort_context)

validation_report = run_preparation_validation(
    source_df=df_canonical,
    cohort_df=cohort_context,
    feature_df=X_model,
    engineered_df=encoded_data,
    distance=D,
)

display(
    pd.DataFrame(
        validation_report.items(),
        columns=["validation_measure", "result"],
    )
)

In [ ]:
# Explicit exclusion checks.
forbidden_model_columns = (
    set(PROFILE_DESCRIPTORS)
    | set(SENSITIVE_EXCLUSIONS)
    | set(SUPPLEMENTARY_FEATURES)
    | {
        "source_row_id",
        "source_index",
        "age_clean",
        "age_imputed",
        "age_was_imputed",
        "age_implausible_flag",
    }
)

assert not (
    set(X_model.columns) & forbidden_model_columns
)

assert len(X_model.columns) == 15
assert X_model.columns.is_unique
assert X_model.index.equals(cohort_context.index)

assert profile_descriptors["source_row_id"].is_unique
assert profile_descriptors["age_imputed"].notna().all()

print("All explicit preparation assertions passed.")

## 17. Persisting Preparation Artefacts

The preparation outputs are stored separately from the original source. The output directory contains the artefacts required to reproduce and audit the prepared analytical representation.

The persisted files include:

- canonical source-to-feature mapping;
- eligible-cohort row manifest;
- primary feature matrix;
- profile descriptors;
- missingness and applicability audits;
- feature decision register;
- category-frequency and encoding audits;
- domain and feature-weight metadata;
- categorical association matrix;
- final dissimilarity matrix;
- validation report;
- source provenance.

The row manifest is kept separate from the feature matrix. This prevents technical identifiers from becoming analytical inputs.

In [ ]:
PREPARATION_DIR = ensure_output_directory(PREPARATION_DIR)
FIGURES_DIR = ensure_output_directory(FIGURES_DIR)

row_mapping = build_respondent_row_mapping(cohort_context)
feature_metadata = build_feature_metadata(DOMAIN_FEATURES)
feature_metadata = feature_metadata.merge(
    domain_membership[
        [
            "feature",
            "domain",
            "domain_feature_count",
            "domain_weight",
            "within_domain_feature_weight",
            "overall_feature_weight",
        ]
    ],
    on=["feature", "domain"],
    how="left",
    suffixes=("_persistence", ""),
)

display(row_mapping.head())
display(feature_metadata)

In [ ]:
outputs = {}

outputs["column_mapping"] = save_dataframe(
    column_mapping_audit,
    PREPARATION_DIR,
    "column_mapping_audit.csv",
)

outputs["eligible_cohort_manifest"] = save_dataframe(
    row_mapping,
    PREPARATION_DIR,
    "eligible_cohort_row_mapping.csv",
)

outputs["model_features"] = save_dataframe(
    X_model,
    PREPARATION_DIR,
    "model_feature_matrix.csv",
)

outputs["profile_descriptors"] = save_dataframe(
    profile_descriptors,
    PREPARATION_DIR,
    "profile_descriptors.csv",
)

outputs["missingness_source"] = save_dataframe(
    source_missingness,
    PREPARATION_DIR,
    "source_missingness_audit.csv",
)

outputs["missingness_cohort"] = save_dataframe(
    cohort_missingness,
    PREPARATION_DIR,
    "cohort_missingness_audit.csv",
)

outputs["applicability"] = save_dataframe(
    applicability_register,
    PREPARATION_DIR,
    "feature_applicability_register.csv",
)

outputs["feature_decisions"] = save_dataframe(
    feature_decision_register,
    PREPARATION_DIR,
    "feature_decision_register.csv",
)

outputs["response_categories"] = save_dataframe(
    response_category_audit,
    PREPARATION_DIR,
    "response_category_audit.csv",
)

outputs["rare_categories"] = save_dataframe(
    rare_category_audit,
    PREPARATION_DIR,
    "rare_category_audit.csv",
)

outputs["encoding_audit"] = save_dataframe(
    encoding_dtype_audit,
    PREPARATION_DIR,
    "encoding_dtype_audit.csv",
)

outputs["domain_metadata"] = save_dataframe(
    domain_membership,
    PREPARATION_DIR,
    "domain_feature_metadata.csv",
)

outputs["association_matrix"] = save_dataframe(
    association_matrix.reset_index(names="feature"),
    PREPARATION_DIR,
    "categorical_association_matrix.csv",
)

outputs["dissimilarity_matrix"] = save_numpy(
    D,
    PREPARATION_DIR,
    "domain_balanced_dissimilarity.npy",
)

outputs["validation_report"] = save_json(
    validation_report,
    PREPARATION_DIR,
    "validation_report.json",
)

outputs["source_metadata"] = save_json(
    source_metadata,
    PREPARATION_DIR,
    "source_metadata.json",
)

outputs["age_imputation_audit"] = save_dataframe(
    age_cleaning_audit,
    PREPARATION_DIR,
    "age_cleaning_audit.csv",
)

outputs["age_quality_summary"] = save_dataframe(
    age_quality_summary,
    PREPARATION_DIR,
    "age_quality_summary.csv",
)

outputs["cohort_flow"] = save_dataframe(
    cohort_flow,
    PREPARATION_DIR,
    "cohort_flow.csv",
)

output_register = pd.DataFrame({
    "artefact": list(outputs.keys()),
    "path": [str(path) for path in outputs.values()],
})

display(output_register)

## 18. Preparation Outcome and Handoff

The Data Preparation phase produces a reproducible analytical dataset from the original OSMI 2016 survey.

The eligible cohort is established using the approved employment and technology-related criteria. The primary matrix contains 15 predeclared workplace-response features and no descriptive, supplementary, sensitive, or technical identifier fields.

Missingness is handled accroding to survey applicability. The primary features are complete int the eligible cohort, so no artificial response imputation is introduced into the primary feature matrix.

Age is cleaned independently as a descriptive variable. The five implausible source values are invalidated in the derived cleaned-age field. Foru such observations occur in the eligible cohort and are imputed using the eliogible cohort's observed median of 32 years. The original age values, cleaned values, imputed values, and imputation indicators remain distinguishable.

Norminal response categories retain their original semantic distinctions. Ordered responses are encoded using nullable integer dtypes. The `mh_as_serious_as_physical` item is treated as nominal because the observed `I don't know` response cannot be placed on the same ordinal scale as `No` and `Yes`.

The final dissimilarity matrix is constructed from the 15 primary features using the approved feature and domain weighting specification. Its structural properties and the feature-level response rules are validated before persistence.

The source file remains unchanged. The saved row manifest preserves the correspondence between prepared records and their source positions without introducing identifiers into the analytical feature matrix.

### Phase Exit Decision

The preparation artefacts are complete when all validation assertions pass and the saved outputs are aligned to the same cohort row order. The resulting feature matrix, profile descriptors, row manifest, metadata, and dissimilarity matrix form the handoff from Data Preparation to the next CRISP-DM phase.

In [ ]:
final_preparation_summary = pd.DataFrame({
    "item": [
        "Source rows",
        "Source columns",
        "Eligible cohort rows",
        "Primary feature count",
        "Feature domains",
        "Age imputation count in eligible cohort",
        "Age median used for imputation",
        "Missing primary feature values",
        "Dissimilarity matrix dimensions",
        "Dissimilarity matrix dtype",
        "Dissimilarity matrix symmetric",
        "Dissimilarity matrix zero diagonal",
        "Validation status",
    ],
    "result": [
        len(df_raw),
        df_raw.shape[1],
        len(cohort_context),
        X_model.shape[1],
        len(DOMAIN_FEATURES),
        int(cohort_context["age_was_imputed"].sum()),
        age_imputation_value,
        int(X_model.isna().sum().sum()),
        str(D.shape),
        str(D.dtype),
        bool(np.allclose(D, D.T, atol=1e-10)),
        bool(np.allclose(np.diag(D), 0.0, atol=1e-10)),
        validation_report["status"],
    ],
})

display(final_preparation_summary)

print("\nPreparation artefacts saved to:")
print(PREPARATION_DIR)